## Open notebook in:
| Colab                                 |  
|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH07/ch07_STORM.ipynb)                                             

#About this notebook

This notebook is implements and evaluates the STORM model, which stands for [Stochastic Transformer-based wORld Model](https://openreview.net/pdf?id=WxnrX42rnS). The STORM model is a reinforcement learning architecture that efficiently combines the sequence modeling power of transformers with the stochastic nature of variational autoencoders to enhance agent performance in complex environments.

Key Features of This Notebook:
- Repository Cloning and Setup: The notebook starts by cloning the official STORM repository, installing necessary dependencies, and setting up the environment for running the model.
- Model Checkpoint Management: It includes downloading and decompressing model checkpoints that are essential for evaluating the STORM model's performance.
- Evaluation Script Creation: The notebook creates a custom Python evaluation script that sets up the environment, loads the trained world model and agent, and evaluates the agent's performance by generating a video of its actions within the environment.
- Execution and Output: The notebook concludes by running the evaluation script and saving the output video, which visualizes the agent's behavior and performance within the selected environment.

Context of STORM:
The STORM model is an advanced approach in model-based reinforcement learning, where a parameterized simulation model of the environment is constructed through self-supervised learning. This model helps the agent improve its policy without constantly relying on real environment samples, making training more efficient.

Performance:
STORM sets a new benchmark with a mean human performance of 126.7% on the Atari 100k benchmark, and its efficient training process makes it highly practical for real-world applications.




# Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/weipu-zhang/STORM.git
!cd STORM
!git checkout e8be598643b4fc0d54e68d870810f05cda5551ec


Cloning into 'STORM'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 73 (delta 27), reused 17 (delta 17), pack-reused 32 (from 2)
Receiving objects: 100% (73/73), 518.51 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Note: switching to 'e8be598643b4fc0d54e68d870810f05cda5551ec'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at e8be598 Update readme.md


In [ ]:
# Change directory to the cloned repo
%cd STORM

/content/STORM/STORM


In [ ]:
!mkdir ckpt

In [ ]:
%ls

agents.py
ckpt/
config_files/
D_TRAJ.7z
env_wrapper.py
eval.py
eval.sh
readme.md
replay_buffer.py
requirements.txt
results/
RoadRunner-life_done-wm_2L512D8H-100k-seed1.tar.gz2bfl3x8v.part
sub_models/
TensorBoard.sh
train.py
train.sh
utils.py


In [ ]:
# Install dependencies
!pip install -r requirements.txt -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.4 MB/s eta 0:00:00


# Download the checkpoints from Google Drive

Here you download the model checkpoints from my training of STORM.

In [ ]:
!gdown 1CKw-viYPAB9BEpbaefLpLeuXHFiDlY3f --output ckpt/RoadRunner-life_done-wm_2L512D8H-100k-seed1.tar.gz

Downloading...
From (original): https://drive.google.com/uc?id=1CKw-viYPAB9BEpbaefLpLeuXHFiDlY3f
From (redirected): https://drive.google.com/uc?id=1CKw-viYPAB9BEpbaefLpLeuXHFiDlY3f&confirm=t&uuid=e76ddf30-bc12-4c6e-a5b3-e60b425e3b98
To: /content/STORM/STORM/ckpt/RoadRunner-life_done-wm_2L512D8H-100k-seed1.tar.gz
100% 3.03G/3.03G [00:52<00:00, 57.7MB/s]


# Decompress the checkpoints

In [ ]:

!tar -xzf ckpt/RoadRunner-life_done-wm_2L512D8H-100k-seed1.tar.gz -C ckpt

# Run Evaluation Script

In [ ]:
# 1. eval.sh: MsPacman -> RoadRunner
!sed -i 's/MsPacman/RoadRunner/g' eval.sh

# 2. train.py / eval.py:
#    shape=image_size -> shape=(image_size, image_size)
!sed -i 's/shape=image_size/shape=(image_size, image_size)/g' train.py eval.py

# 3. train.py / eval.py:
#    AsyncVectorEnv に SameStep を指定
!sed -i \
  's/gymnasium\.vector\.AsyncVectorEnv(env_fns=env_fns)/gymnasium.vector.AsyncVectorEnv(env_fns=env_fns, autoreset_mode="SameStep")/g' \
  train.py eval.py

# 4. ALE を明示的に登録
!sed -i '/^import gymnasium$/a import ale_py\ngymnasium.register_envs(ale_py)' train.py eval.py

In [ ]:
!mkdir -p eval_result

# Make the eval.sh script executable
!chmod +x eval.sh

# Run the eval.sh script
!./eval.sh

Namespace(config_path='config_files/STORM.yaml', env_name='ALE/RoadRunner-v5', run_name='RoadRunner-life_done-wm_2L512D8H-100k-seed1')
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
[100000]
  0% 0/1 [00:00<?, ?it/s]Current env: ALE/RoadRunner-v5
Mean reward: 16665.0
100% 1/1 [03:02<00:00, 182.33s/it]


In [ ]:
# Create the evaluation script
eval_script = """
import argparse
from collections import deque

import ale_py
import gymnasium
import imageio
import numpy as np
import torch
from einops import rearrange

import agents
import env_wrapper
from sub_models.world_models import WorldModel
from utils import load_config


gymnasium.register_envs(ale_py)


def build_single_env(env_name: str, image_size: int):
    env = gymnasium.make(
        env_name,
        full_action_space=False,
        render_mode="rgb_array",
        frameskip=1,
    )

    # STORMと同じAtari前処理
    env = env_wrapper.MaxLast2FrameSkipWrapper(env, skip=4)

    # Gymnasium 1.xではtupleを指定
    env = gymnasium.wrappers.ResizeObservation(
        env,
        shape=(image_size, image_size),
    )

    return env


def evaluate(
    env,
    world_model: WorldModel,
    agent: agents.ActorCriticAgent,
    video_path: str,
    seed: int = 0,
):
    world_model.eval()
    agent.eval()

    env.action_space.seed(seed)
    obs, _ = env.reset(seed=seed)

    total_reward = 0.0
    frames = []

    # STORM公式eval.pyと同じ最大16ステップのコンテキスト
    context_obs = deque(maxlen=16)
    context_action = deque(maxlen=16)

    terminated = False
    truncated = False

    while not (terminated or truncated):
        # render_mode="rgb_array"で取得
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        # (H, W, C)
        #   ->
        # (1, 1, C, H, W)
        obs_tensor = rearrange(
            torch.as_tensor(
                obs,
                dtype=torch.float32,
                device="cuda",
            ),
            "H W C -> 1 1 C H W",
        ) / 255.0

        with torch.inference_mode():
            if len(context_action) == 0:
                # STORM公式コードと同様、最初の1stepはrandom action
                action = env.action_space.sample()

            else:
                # context_obs:
                #   list[(1, 1, C, H, W)]
                #
                # ->
                #
                #   (1, L, C, H, W)
                context_obs_tensor = torch.cat(
                    list(context_obs),
                    dim=1,
                )

                # 観測をstochastic latentへ変換
                context_latent = world_model.encode_obs(
                    context_obs_tensor
                )

                # context_action:
                #   [a0, a1, ..., a_{L-1}]
                #
                # ->
                #
                #   (1, L)
                context_action_tensor = torch.tensor(
                    list(context_action),
                    dtype=torch.float32,
                    device="cuda",
                ).unsqueeze(0)

                # Transformerから
                #   stochastic latent
                #   deterministic feature
                # を取得
                prior_flattened_sample, last_dist_feat = (
                    world_model.calc_last_dist_feat(
                        context_latent,
                        context_action_tensor,
                    )
                )

                # Actorへの入力:
                #
                # [stochastic latent, transformer feature]
                agent_input = torch.cat(
                    [
                        prior_flattened_sample,
                        last_dist_feat,
                    ],
                    dim=-1,
                )

                action_array = agent.sample_as_env_action(
                    agent_input,
                    greedy=False,
                )

                action = int(action_array[0])

        # 現時点の (obs_t, action_t) をcontextに追加
        context_obs.append(obs_tensor)
        context_action.append(action)

        obs, reward, terminated, truncated, _ = env.step(action)

        total_reward += float(reward)

    env.close()

    if frames:
        imageio.mimsave(
            video_path,
            frames,
            fps=30,
        )

    return total_reward


def build_world_model(conf, action_dim: int):
    return WorldModel(
        in_channels=conf.Models.WorldModel.InChannels,
        action_dim=action_dim,
        transformer_max_length=conf.Models.WorldModel.TransformerMaxLength,
        transformer_hidden_dim=conf.Models.WorldModel.TransformerHiddenDim,
        transformer_num_layers=conf.Models.WorldModel.TransformerNumLayers,
        transformer_num_heads=conf.Models.WorldModel.TransformerNumHeads,
    ).cuda()


def build_agent(conf, action_dim: int):
    return agents.ActorCriticAgent(
        feat_dim=32 * 32 + conf.Models.WorldModel.TransformerHiddenDim,
        num_layers=conf.Models.Agent.NumLayers,
        hidden_dim=conf.Models.Agent.HiddenDim,
        action_dim=action_dim,
        gamma=conf.Models.Agent.Gamma,
        lambd=conf.Models.Agent.Lambda,
        entropy_coef=conf.Models.Agent.EntropyCoef,
    ).cuda()


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "-n",
        type=str,
        required=True,
    )
    parser.add_argument(
        "-config_path",
        type=str,
        required=True,
    )
    parser.add_argument(
        "-env_name",
        type=str,
        required=True,
    )
    parser.add_argument(
        "-checkpoint_path",
        type=str,
        required=True,
    )
    parser.add_argument(
        "-video_path",
        type=str,
        required=True,
        help="Path to save the output video",
    )
    parser.add_argument(
        "-step",
        type=int,
        default=100000,
        help="Checkpoint step to load",
    )
    parser.add_argument(
        "-seed",
        type=int,
        default=0,
    )

    args = parser.parse_args()

    conf = load_config(args.config_path)

    env = build_single_env(
        args.env_name,
        conf.BasicSettings.ImageSize,
    )

    action_dim = env.action_space.n

    world_model = build_world_model(
        conf,
        action_dim,
    )

    agent = build_agent(
        conf,
        action_dim,
    )

    world_model_path = (
        f"{args.checkpoint_path}/world_model_{args.step}.pth"
    )
    agent_path = (
        f"{args.checkpoint_path}/agent_{args.step}.pth"
    )

    print(f"Loading world model: {world_model_path}")
    print(f"Loading agent:       {agent_path}")

    world_model.load_state_dict(
        torch.load(
            world_model_path,
            map_location="cuda",
            weights_only=True,
        )
    )

    agent.load_state_dict(
        torch.load(
            agent_path,
            map_location="cuda",
            weights_only=True,
        )
    )

    total_reward = evaluate(
        env=env,
        world_model=world_model,
        agent=agent,
        video_path=args.video_path,
        seed=args.seed,
    )

    print(f"Total reward: {total_reward}")


if __name__ == "__main__":
    main()
   """

with open("eval.py", "w") as file:
    file.write(eval_script)

# Step 10: Run the evaluation script and save the video
!python eval.py -n STORM_eval -config_path config_files/STORM.yaml -env_name ALE/RoadRunner-v5 -checkpoint_path ckpt/RoadRunner-life_done-wm_2L512D8H-100k-seed1 -video_path output_video.mp4


A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
/content/STORM/STORM/sub_models/world_models.py:271: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)
/content/STORM/STORM/agents.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)
Loading world model: ckpt/RoadRunner-life_done-wm_2L512D8H-100k-seed1/world_model_100000.pth
Loading agent:       ckpt/RoadRunner-life_done-wm_2L512D8H-100k-seed1/agent_100000.pth
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the

In [ ]:
from IPython.display import Video

Video("output_video.mp4", embed=True)